In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

from llm import qwen_utils, tools

# Data paths
DATA_ROOT = Path("data/preprocessed/qwen3")
CLIP_ID = "video01_00240"  # example clip with qwen3 features

/home/guests/nicolas_stellwag/surgery-scene-graphs/.pixi/envs/default/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Load Qwen3 patch features for a single frame
# These are stored as (N, 16384) = [main | ds0 | ds1 | ds2] concatenated features
clip_dir = DATA_ROOT / CLIP_ID
patch_feat_dir = clip_dir / "qwen3_patch_features"
instance_feat_dir = clip_dir / "qwen3_instance_features"
image_dir = clip_dir / "images"

# Load first frame's patch features for demo
frame_idx = 0
patch_feats = np.load(patch_feat_dir / f"{frame_idx:06d}_f.npy")  # (N, 16384)
print(f"Patch features shape: {patch_feats.shape}")
print(f"Feature dim: {patch_feats.shape[-1]} = 4096 * 4 (main + 3 deepstack)")

# Also load instance features for comparison
instance_feats = np.load(instance_feat_dir / f"{frame_idx:06d}_f.npy")  # (num_instances, 16384)
print(f"Instance features shape: {instance_feats.shape}")

Patch features shape: (224, 16384)
Feature dim: 16384 = 4096 * 4 (main + 3 deepstack)
Instance features shape: (7, 16384)


In [3]:
# Define tools for the agentic loop
# Tools are Dict[name -> (callable, json_spec)]
available_tools = {
    "node_distances_through_time": (tools.node_distances_through_time, tools.spec_node_distances_through_time),
    "weather_munich": (tools.weather_munich, tools.spec_weather_munich)
}

# For demo purposes, let's use just the weather tool
demo_tools = {
    "weather_munich": (tools.weather_munich, tools.spec_weather_munich)
}

In [4]:
# Load patched Qwen3-VL model with custom feature injection support
model, processor = qwen_utils.get_patched_qwen3()
print(f"Model loaded on device: {model.device}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on device: cuda:0


In [5]:
# Example 1: Simple generation with vision features (no tools)
# Using the full image patch features

vision_features = [torch.from_numpy(patch_feats)]

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful medical assistant analyzing cholecystectomy surgery images."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": None},
            {"type": "text", "text": "Describe what you see in this surgical image."},
        ],
    },
]

response = qwen_utils.generate_with_vision_features(
    messages=messages,
    vision_features=vision_features,
    model=model,
    processor=processor,
    qwen_version="qwen3",
    max_tokens=256,
)
print("Response:", response)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Response: This is a close-up intraoperative surgical image, likely from a laparoscopic cholecystectomy (gallbladder removal surgery).

Here’s a detailed description of what is visible:

- **The Gallbladder**: The central focus is the gallbladder, which appears as a pale, yellowish, pear-shaped organ. Its surface is somewhat translucent and shows visible blood vessels (appearing as thin, reddish lines).

- **Surgical Instruments**: A metallic surgical instrument, likely a laparoscopic grasper or clamp, is visible in the lower-left portion of the image. It is holding or manipulating the gallbladder or surrounding tissue.

- **Surrounding Anatomy**: The gallbladder is nestled against surrounding reddish, fleshy tissue, which is likely the liver (to the right) and possibly the duodenum or other abdominal structures. The tissue appears moist and glistening, consistent with the internal abdominal cavity.

- **Surgical Field**: The image is taken from a laparoscopic perspective, meaning it’s 

In [61]:
graph_root = Path("output/qwen3/video01_00240/graph_strongfilter")
feats = np.load(graph_root / "c_qwen_feats.npz")
adjacencies = np.load(graph_root / "graph.npy")
centers = np.load(graph_root / "c_centers.npy")
centroids = np.load(graph_root / "c_centroids.npy")
extents = np.load(graph_root / "c_extents.npy")
positions = np.load(graph_root / "positions.npy")

In [62]:
from functools import partial
from llm.tools import ALL_TOOLS
import pprint

In [63]:
ALL_TOOLS["node_distances_through_time"] = (
    partial(
        ALL_TOOLS["node_distances_through_time"][0],
        positions=positions,
        node_centroids=centroids,
    ),
    ALL_TOOLS["node_distances_through_time"][1],
)

In [64]:
response = qwen_utils.prompt_with_graph_at_timestep(
    question="During which timesteps is the grasper actively grasping the gallbladder?",
    node_feats=feats,
    timestep_idx=0,
    adjacency_matrices=adjacencies,
    node_centers=centers,
    node_centroids=centroids,
    node_extents=extents,
    model=model,
    processor=processor,
    qwen_version="qwen3",
    system_prompt="You are a surgical scene understanding assistant analyzing a cholecystectomy procedure. You can use tools to answer the user's questions.",
    tools=ALL_TOOLS,
)
pprint.pprint(response["message_history"])

[{'content': [{'text': 'You are a surgical scene understanding assistant '
                       'analyzing a cholecystectomy procedure. You can use '
                       "tools to answer the user's questions.",
               'type': 'text'}],
  'role': 'system'},
 {'content': [{'text': '<spatial-graph>\n', 'type': 'text'},
              {'text': '<node id="0">\n', 'type': 'text'},
              {'text': '<descriptor>', 'type': 'text'},
              {'image': None, 'type': 'image'},
              {'text': '</descriptor>\n', 'type': 'text'},
              {'text': '<centroid x="-0.08" y="-0.05" z="0.71"/>\n',
               'type': 'text'},
              {'text': '<extent x="0.18" y="0.16" z="0.18"/>\n',
               'type': 'text'},
              {'text': '</node>\n', 'type': 'text'},
              {'text': '<node id="1">\n', 'type': 'text'},
              {'text': '<descriptor>', 'type': 'text'},
              {'image': None, 'type': 'image'},
              {'text': '</descrip